# Cross-exchange SPDR S&P 500 ETF: SPY5.L / SPY5z.CHIX / SPY5.P 
SPY5.L   + SPY5z.CHIX + SPY5.P  


Workflow:
- data loading
- percentage log returns
- EDA (time evolution, distribution, PACF, ACF)
- Ljung box test based on the combinations of p,q found in ACF and PACF --) ARMA model implied
- Rolling window or expanding window for best p,q combinations of ARMA evaluated on AIC,SBIC,HQIC information criterion on test set


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, skew

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from statsmodels.tsa.api import VAR
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch

from sklearn.metrics import mean_squared_error

from tqdm import tqdm
import warnings


## Loading in Data and EDA

In [ ]:
df_snp = pd.read_csv('output/sp_g21.csv')
df_snp['DateTime'] = pd.to_datetime(df_snp['DateTime']) # Ensure DateTime is in datetime format
print(df_snp.isnull().sum()) # no missing values

print(df_snp["symbol"].value_counts())


dfSPY_p = df_snp[df_snp['symbol'] == 'SPY5.L'].copy()
dfSPY_p['log_returns'] = 100 *  np.log(dfSPY_p['price'] / dfSPY_p['price'].shift(1)) # Log returns in percentage
dfSPY_p = dfSPY_p.dropna() # Drop NA values resulting from the shift operation
print(dfSPY_p.describe())
print(dfSPY_p.head())

# Plot the evolution of prices for each symbol

plt.figure(figsize=(15, 7))
sns.lineplot(data=df_snp, x='DateTime', y='price', hue='symbol')
plt.title('Price Evolution of Selected S&P 500 Symbols (2024-2025)')

plt.xlabel('Date')
plt.ylabel('Price')
plt.legend(title='Symbol')

plt.show()

In [ ]:

# Interactive plot 

fig = px.line(df_snp, x='DateTime', y='price', color='symbol', 
              title='Interactive Price Evolution of S&P 500 Symbols')
fig.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(dfSPY_p['log_returns'], ax=axes[0], lags=20, alpha=0.05)
plot_pacf(dfSPY_p['log_returns'], ax=axes[1], lags=20, method='ywm', alpha=0.05)

axes[0].set_title("ACF Plot with 95% Confidence Interval (Shaded Area)")
axes[1].set_title("PACF Plot with 95% Confidence Interval (Shaded Area)")
plt.show()

In [ ]:
dfSPY_p['DateTime'] = pd.to_datetime(dfSPY_p['DateTime'])
dfSPY_p.set_index('DateTime', inplace=True)

price_resampled = dfSPY_p['price'].resample('5min').last().dropna()

# Resample to 5-minute frequency
log_returns_resampled = 100 * np.log(price_resampled / price_resampled.shift(1))
clean_log_returns = log_returns_resampled.dropna()

# Plot the ACF and PACF of the resampled series
print("--- Diagnostics for Resampled Data ---")
print(clean_log_returns.describe())
print("------------------------------------")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(clean_log_returns, ax=axes[0], lags=10, alpha=0.05)
plot_pacf(clean_log_returns, ax=axes[1], lags=10, alpha=0.05)
axes[0].set_ylim(-0.1, 0.1)
axes[1].set_ylim(-0.1, 0.1)
axes[0].set_title("ACF Plot")
plt.show()

In [ ]:
# Skewness and Kurtosis
skewness = skew(clean_log_returns)
kurt = kurtosis(clean_log_returns)
print(f"Skewness: {skewness:.4f}")
print(f"Kurtosis: {kurt:.4f}")
# Plot distribution of log returns
plt.figure(figsize=(10, 5))
sns.histplot(clean_log_returns, bins=50, kde=True)
plt.title('Distribution of 5-Minute Log Returns for SPY5.L')
plt.xlabel('Log Returns (%)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Calculate the percentage of outliers in this sample beyond 3 standard deviations
std_dev = clean_log_returns.std()
outliers = clean_log_returns[(clean_log_returns > 3 * std_dev) | (clean_log_returns < -3 * std_dev)]
outlier_percentage = (len(outliers) / len(clean_log_returns)) * 100 
print(f"Percentage of outliers beyond 3 standard deviations: {outlier_percentage:.4f}%")

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

# Run the Ljung-Box test on the log returns
# We can test for autocorrelation up to, for example, 10 lags
ljung_box_results = acorr_ljungbox(clean_log_returns, lags=[10], return_df=True)

print(ljung_box_results)

#### LOOP for ARMA(p,q) to find optimal order up to 10,10

In [ ]:
max_order = 10 # Maximum order for p and q to test
results = []

for p in range(max_order + 1):
    for q in range(max_order + 1):
        try:
            # Fit an ARMA(p,q) model (which is an ARIMA(p,0,q))
            model = ARIMA(clean_log_returns, order=(p, 0, q)).fit()
            
            # Store the results
            results.append({
                'p': p,
                'q': q,
                'AIC': model.aic,
                'BIC': model.bic,
                'HQIC': model.hqic
            })
        except Exception as e:
            # Handle cases where the model fails to converge
            print(f"Failed to fit ARMA({p},{q}). Error: {e}")
            continue

# Create a DataFrame from the results for easy viewing
results_df = pd.DataFrame(results)

print("--- Information Criteria for ARMA(p,q) Models ---")
print(results_df)

# Find the optimal (p,q) for each criterion
optimal_aic = results_df.loc[results_df['AIC'].idxmin()]
optimal_bic = results_df.loc[results_df['BIC'].idxmin()]
optimal_hqic = results_df.loc[results_df['HQIC'].idxmin()]

print("\n--- Optimal Model Orders ---")
print(f"Optimal according to AIC: ARMA({int(optimal_aic['p'])}, {int(optimal_aic['q'])})")
print(f"Optimal according to BIC: ARMA({int(optimal_bic['p'])}, {int(optimal_bic['q'])})")
print(f"Optimal according to HQIC: ARMA({int(optimal_hqic['p'])}, {int(optimal_hqic['q'])})")

In [ ]:


# --- Choose final model based on the criteria above (e.g., BIC) ---
best_p = int(optimal_bic['p'])
best_q = int(optimal_bic['q'])

print(f"\n--- Fitting final ARMA({best_p},{best_q}) model for diagnostics ---")
final_model = ARIMA(clean_log_returns, order=(best_p, 0, best_q)).fit()

# Get the model residuals
residuals = final_model.resid

# --- Perform the Ljung-Box test on the residuals ---
# Check for autocorrelation up to a reasonable number of lags (e.g., 20)
ljung_box_results = acorr_ljungbox(residuals, lags=[20], return_df=True)

print("\n--- Ljung-Box Test on Model Residuals ---")
print(ljung_box_results)

# Interpretation
p_value = ljung_box_results['lb_pvalue'].values[0]
if p_value > 0.05:
    print(f"\nP-value ({p_value:.4f}) is greater than 0.05.")
    print("Conclusion: The residuals appear to be independent (Good Fit).")
else:
    print(f"\nP-value ({p_value:.4f}) is less than 0.05.")
    print("Conclusion: There is significant autocorrelation in the residuals (Bad Fit).")

In [ ]:


# --- The model suggested by AIC ---
best_p = 4
best_q = 3

print(f"\n--- Fitting the ARMA({best_p},{best_q}) model suggested by AIC ---")
final_model_aic = ARIMA(clean_log_returns, order=(best_p, 0, best_q)).fit()

# Get the residuals from this new model
residuals_aic = final_model_aic.resid

# --- Perform the Ljung-Box test on the new residuals ---
ljung_box_results_aic = acorr_ljungbox(residuals_aic, lags=[20], return_df=True)

print("\n--- Ljung-Box Test on ARMA(4,3) Model Residuals ---")
print(ljung_box_results_aic)

# Interpretation
p_value = ljung_box_results_aic['lb_pvalue'].values[0]
if p_value > 0.05:
    print(f"\nP-value ({p_value:.4f}) is greater than 0.05.")
    print("Conclusion: The residuals appear to be independent (Good Fit).")
else:
    print(f"\nP-value ({p_value:.4f}) is less than 0.05.")
    print("Conclusion: There is significant autocorrelation in the residuals (Bad Fit).")

## Further EDA

In [ ]:

# Resample 5minute log returns to 1-hour by summing them
hourly_log_returns = clean_log_returns.resample('1H').sum().dropna()

# Calculate the percentage of outliers for the hourly data ---
hourly_std_dev = hourly_log_returns.std()
hourly_outliers = hourly_log_returns[(hourly_log_returns > 3 * hourly_std_dev) | (hourly_log_returns < -3 * hourly_std_dev)]
hourly_outlier_percentage = (len(hourly_outliers) / len(hourly_log_returns)) * 100 
print(f"\nPercentage of hourly outliers beyond 3 std devs: {hourly_outlier_percentage:.4f}%")

# Kurtosis and Skewness
hourly_kurtosis = kurtosis(hourly_log_returns)
hourly_skewness = skew(hourly_log_returns)
print(f"\nDaily Log Returns Kurtosis: {hourly_kurtosis:.4f}")
print(f"Daily Log Returns Skewness: {hourly_skewness:.4f}")


# --- 3. Plot the combined distribution of all hourly returns ---
plt.figure(figsize=(10, 6))
sns.histplot(hourly_log_returns, bins=50, kde=True)
plt.title('Distribution of 1-Hour Log Returns for SPY5.L (Combined)')
plt.xlabel('Hourly Log Returns (%)')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:

# Resample 5minute log returns to 1-hour by summing them
daily_log_returns = clean_log_returns.resample('1D').sum().dropna()

# Calculate the percentage of outliers for the daily data ---
daily_std = daily_log_returns.std()
daily_outliers = daily_log_returns[(daily_log_returns > 3 * daily_std) | (daily_log_returns < -3 * daily_std)]
daily_outlier_percentage = (len(daily_outliers) / len(daily_log_returns)) * 100
print(f"\nPercentage of daily outliers beyond 3 std devs: {daily_outlier_percentage:.4f}%")

# Kurtosis and Skewness
daily_kurtosis = kurtosis(daily_log_returns)
daily_skewness = skew(daily_log_returns)
print(f"\nDaily Log Returns Kurtosis: {daily_kurtosis:.4f}")
print(f"Daily Log Returns Skewness: {daily_skewness:.4f}")

# --- 3. Plot the combined distribution of all hourly returns ---
plt.figure(figsize=(10, 6))
sns.histplot(daily_log_returns, bins=50, kde=True)
plt.title('Distribution of Daily Log Returns for SPY5.L (Combined)')
plt.xlabel('Hourly Log Returns (%)')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# This assumes 'clean_log_returns' is your 5-minute log return Series.

# --- 1. Calculate Weekly Mean and Std Dev for Every Data Point ---
# The transform() method is powerful: it performs a groupby calculation
# but returns a Series with the original index.
weekly_mean = clean_log_returns.groupby(pd.Grouper(freq='W')).transform('mean')
weekly_std = clean_log_returns.groupby(pd.Grouper(freq='W')).transform('std')

# --- 2. Define Outlier Bounds Based on Weekly Stats ---
upper_bound = weekly_mean + 3 * weekly_std
lower_bound = weekly_mean - 3 * weekly_std

# --- 3. Identify all points that are outliers relative to their own week ---
outliers = clean_log_returns[(clean_log_returns > upper_bound) | (clean_log_returns < lower_bound)]

print(f"Identified {len(outliers)} outliers across the entire sample period.")

# --- 4. Plot the Entire Time Series with All Outliers Highlighted ---
plt.figure(figsize=(18, 8))

# Plot the main time series
plt.plot(clean_log_returns.index, clean_log_returns, label='Log Returns', color='b', zorder=1, linewidth=0.5)

# Plot the identified outliers as red scatter points on top
plt.scatter(outliers.index, outliers, color='r', label='Weekly Outliers (>3 Std Dev)', zorder=2, s=10)

plt.title('5-Minute Log Returns with Outliers Identified on a Weekly Basis')
plt.xlabel('Date')
plt.ylabel('Log Returns (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Let's use the ARMA(2,2) model which was a strong candidate.
p, q = 2, 2
model_name = f'ARMA({p},{q})'

# --- 1. Split Data into Training and a 1-Hour Test Set ---
test_horizon = 12 # 12 periods * 5 minutes = 60 minutes
train_data = clean_log_returns[:-test_horizon]
test_data = clean_log_returns[-test_horizon:]

# --- 2. Fit the ARMA Model on the Full Training Set ---
print(f"Fitting {model_name} on all available training data...")
model = ARIMA(train_data, order=(p, 0, q))
fitted_model = model.fit()

# --- 3. Forecast the Next 12 Steps (1 Hour) ---
forecast = fitted_model.forecast(steps=test_horizon)

# --- 4. Plot the Forecast vs. Actual Values ---
plt.figure(figsize=(12, 6))
plt.plot(test_data.index, test_data, label='Actual Returns', marker='o')
plt.plot(test_data.index, forecast, label='Forecasted Returns', linestyle='--', marker='x')
plt.title(f'{model_name} Forecast vs. Actuals for the Next Hour')
plt.xlabel('Time')
plt.ylabel('Log Returns (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Section 3.2

In [ ]:
# Split Data into Train, Holdout, and Test Sets 
test_size = 12  # Last 1 hour for the final test set
holdout_size = 12*3 # next 3hours

test_data = clean_log_returns[-test_size:]
holdout_data = clean_log_returns[-(holdout_size + test_size):-test_size]
train_data = clean_log_returns[:-(holdout_size + test_size)]

print(f"Data Split:")
print(f"Training set size:   {len(train_data)}")
print(f"Holdout set size:    {len(holdout_data)}")
print(f"Test set size:       {len(test_data)}")
print("-" * 40)


# --- 2. In-Sample Identification on Training Data ---
# Define the (p,q) combinations to test
combinations_to_test = [(0,0),(0,1), (0,2),
                        (1,0), (2,0), (1,1),
                        (1,2), (2,1), (2,2)]

results_list = []

for p, q in tqdm(combinations_to_test, desc="Fitting In-Sample Models"):
    model_name = f'ARMA({p},{q})'
    try:
        model = ARIMA(train_data, order=(p, 0, q)).fit()
        lb_test = acorr_ljungbox(model.resid, lags=[20], return_df=True)
        results_list.append({
            'Model': model_name, 'p': p, 'q': q, 'Log_likelihood': model.llf, 'AIC': model.aic, 'BIC': model.bic, 'HQIC': model.hqic, 
            'Ljung-Box p-value': lb_test['lb_pvalue'].values[0]
        })
    except Exception as e:
        continue
summary_df = pd.DataFrame(results_list).sort_values(by='BIC')
print("\n--- In-Sample Model Comparison (on Training Data) ---")
print(summary_df)
print("-" * 40)

# --- 3. Expanding Window Validation on Holdout Set ---
holdout_mse_results = {}

# --- WRAP THE LOOP WITH TQDM FOR A PROGRESS BAR ---
for p, q in tqdm(combinations_to_test, desc="Validating on Holdout Set"):
    model_name = f'ARMA({p},{q})'
    history = list(train_data)
    predictions = []
    
    for t in range(len(holdout_data)):
        model = ARIMA(history, order=(p, 0, q))
        model_fit = model.fit()
        yhat = model_fit.forecast()[0]
        predictions.append(yhat)
        history.append(holdout_data.iloc[t])
        
    mse = mean_squared_error(holdout_data, predictions)
    holdout_mse_results[model_name] = mse

# --- 4. Compare Out-of-Sample Performance on Holdout Set ---
print("\n--- Expanding Window Holdout Set Results (Lower MSE is Better) ---")
for model, mse in sorted(holdout_mse_results.items(), key=lambda item: item[1]):
    print(f"Model: {model}, Holdout MSE: {mse:.8f}")

In [ ]:
# Choose the Single Best Model Based on Holdout Performance ---
best_model_name = min(holdout_mse_results, key=holdout_mse_results.get)
best_model_mse = holdout_mse_results[best_model_name]
best_p, best_q = eval(best_model_name.replace('ARMA', '')) # Extracts (p,q) from the string name

print("\n--- Model Selection Results (from Holdout Set) ---")
print(f"The best model is {best_model_name} with a Holdout MSE of {best_model_mse:.8f}")
print("-" * 50)
# Final Evaluation: Train on ALL data before the test set 
# We combine the original training and holdout data for the final fit
final_train_data = clean_log_returns[:-test_size]

print(f"Fitting the best model ({best_model_name}) on all {len(final_train_data)} pre-test observations...")
final_model = ARIMA(final_train_data, order=(best_p, 0, best_q)).fit()

# Forecast and Evaluate on the Unseen Test Set 
final_forecast = final_model.forecast(steps=test_size)
final_test_mse = mean_squared_error(test_data, final_forecast)

print(f"\n--- Final Model Performance (on Test Set) ---")
print(f"Final Test MSE for {best_model_name}: {final_test_mse:.8f}")

# Plot final forecast vs. actual test data
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(test_data.index, test_data, label='Actual Test Values', marker='o')
plt.plot(test_data.index, final_forecast, label='Final Forecast', linestyle='--', marker='x')
plt.title(f'Final Forecast vs. Actuals for {best_model_name}')
plt.legend()
plt.show()

# Section 3.3 - VAR vs Cointegration

In [ ]:
print(df_snp["symbol"].value_counts())


In [ ]:
# Create a wide-format dataframe for the 3 symbols
your_symbols = ['SPY5.L', 'SPY5z.CHIX', 'SPY5.P']
df_wide = df_snp.pivot(index='DateTime', columns='symbol', values='price')[your_symbols]


In [ ]:
df_wide["SPY5z.CHIX"].isna().sum()

In [ ]:
df_wide.describe()

In [ ]:
# plot the price evolution of the three symbols
plt.figure(figsize=(15, 7))
sns.lineplot(data=df_wide)
plt.title('Price Evolution of SPY5.L, SPY5z.CHIX, and SPY5.P (2024-2025)')
plt.xlabel('DateTime')
plt.ylabel('Price')
plt.legend(title='Symbol')
plt.show()



In [ ]:
warnings.filterwarnings("ignore")


your_symbols = ['SPY5.L', 'SPY5z.CHIX', 'SPY5.P']
df_wide = df_snp.pivot(index='DateTime', columns='symbol', values='price')[your_symbols]

# --- SOLUTION TO MemoryError: RESAMPLE TO A LOWER FREQUENCY (e.g., 30 minutes) ---
resample_freq = '30min'
df_resampled = df_wide.resample(resample_freq).last().ffill().dropna()
print(f"Data has been resampled to {resample_freq} frequency. New shape: {df_resampled.shape}")

# Create the two datasets we need from the resampled data
log_prices = np.log(df_resampled)
log_returns = 100 * log_prices.diff().dropna()
print("-" * 50)


# --- 1. Test for Stationarity (Unit Root Tests) ---
print("\n--- Augmented Dickey-Fuller (ADF) Unit Root Tests ---")
print("\nTesting Log-Prices:")
for name, series in log_prices.items():
    result = adfuller(series)
    print(f'ADF test for {name}: p-value = {result[1]:.4f}. Result: {"Stationary" if result[1] <= 0.05 else "Non-Stationary (I(1))"}')

print("\nTesting Log-Returns:")
for name, series in log_returns.items():
    result = adfuller(series)
    print(f'ADF test for {name}: p-value = {result[1]:.4f}. Result: {"Stationary" if result[1] <= 0.05 else "Non-Stationary"}')
print("-" * 50)


# --- 2. VAR Model on Log-Returns ---
print("\n--- Part 1: Vector Autoregression (VAR) on Log-Returns ---")
var_model = VAR(log_returns)
lag_selection_results = var_model.select_order(maxlags=15)
optimal_lags = lag_selection_results.bic
print(lag_selection_results.summary())
print(f"\nOptimal lag order chosen by BIC: {optimal_lags}")

var_results = var_model.fit(optimal_lags)
print("\n--- VAR Model Summary ---")
print(var_results.summary())

print("\n--- VAR Residual Diagnostics ---")
whiteness_test_lags = optimal_lags + 5
whiteness_test = var_results.test_whiteness(nlags=whiteness_test_lags)
print(whiteness_test.summary())
print("-" * 50)


# --- 3. Cointegration and VECM Model on Log-Prices ---
print("\n--- Part 2: Cointegration and VECM on Log-Prices ---")
johansen_test = coint_johansen(log_prices, det_order=0, k_ar_diff=optimal_lags - 1)

def johansen_interpretation(test, alpha=0.05):
    crit_vals_col = {0.1: 0, 0.05: 1, 0.01: 2}[alpha]
    trace_stat = test.lr1
    crit_vals = test.cvt[:, crit_vals_col]
    print(f"\nTrace Statistics vs {1-alpha:.0%} Critical Values:")
    for r in range(len(trace_stat)):
        print(f"H0: rank <= {r}: Trace Stat = {trace_stat[r]:.2f}, Crit Val = {crit_vals[r]:.2f} -> {'Reject H0' if trace_stat[r] > crit_vals[r] else 'Fail to Reject H0'}")

johansen_interpretation(johansen_test)
# Based on theory for 3 variables, the maximum rank is 2.
vecm_rank = 2
print(f"\nProceeding with a cointegration rank of {vecm_rank}.")


vecm_model = VECM(log_prices, k_ar_diff=optimal_lags-1, coint_rank=vecm_rank, deterministic='n')
vecm_results = vecm_model.fit()
print("\n--- VECM Summary ---")
print(vecm_results.summary())
print("-" * 50)


# --- 4. Post-Estimation Analysis ---
print("\n--- Part 3: Impulse Response and Variance Decomposition ---")
irf = var_results.irf(periods=20)
irf.plot(orth=True)
plt.suptitle('Impulse Response Functions', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

fevd = var_results.fevd(20)
print("\n--- Forecast Error Variance Decomposition ---")
print(fevd.summary())
fevd.plot()
plt.suptitle('Forecast Error Variance Decomposition', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
warnings.filterwarnings("ignore")


your_symbols = ['SPY5.L', 'SPY5z.CHIX', 'SPY5.P']
df_wide = df_snp.pivot(index='DateTime', columns='symbol', values='price')[your_symbols]

# SOLUTION TO MemoryError: RESAMPLE TO A LOWER FREQUENCY 
resample_freq = '30min'
df_resampled = df_wide.resample(resample_freq).last().ffill().dropna()
print(f"Data has been resampled to {resample_freq} frequency. New shape: {df_resampled.shape}")

# Create the two datasets we need from the resampled data
log_prices = np.log(df_resampled)
log_returns = 100 * log_prices.diff().dropna()
print("-" * 50)


# Test for Stationarity (Unit Root Tests)
print("\n--- Augmented Dickey-Fuller (ADF) Unit Root Tests ---")
print("\nTesting Log-Prices:")
for name, series in log_prices.items():
    result = adfuller(series)
    print(f'ADF test for {name}: p-value = {result[1]:.4f}. Result: {"Stationary" if result[1] <= 0.05 else "Non-Stationary (I(1))"}')

print("\nTesting Log-Returns:")
for name, series in log_returns.items():
    result = adfuller(series)
    print(f'ADF test for {name}: p-value = {result[1]:.4f}. Result: {"Stationary" if result[1] <= 0.05 else "Non-Stationary"}')
print("-" * 50)


# VAR Model on Log-Returns 
print("\n--- Part 1: Vector Autoregression (VAR) on Log-Returns ---")
var_model = VAR(log_returns)
lag_selection_results = var_model.select_order(maxlags=15)
optimal_lags = lag_selection_results.bic
print(lag_selection_results.summary())
print(f"\nOptimal lag order chosen by BIC: {optimal_lags}")

var_results = var_model.fit(optimal_lags)
print("\n--- VAR Model Summary ---")
print(var_results.summary())

print("\n--- VAR Residual Diagnostics ---")
# Testing the residuals using white test
whiteness_test_lags = optimal_lags + 5
whiteness_test = var_results.test_whiteness(nlags=whiteness_test_lags)
print(whiteness_test.summary())
print("-" * 50)


# Cointegration and VECM Model on Log-Prices 
print("\n--- Part 2: Cointegration and VECM on Log-Prices ---")
johansen_test = coint_johansen(log_prices, det_order=0, k_ar_diff=optimal_lags - 1)

def johansen_interpretation(test, alpha=0.05):
    crit_vals_col = {0.1: 0, 0.05: 1, 0.01: 2}[alpha]
    trace_stat = test.lr1
    crit_vals = test.cvt[:, crit_vals_col]
    print(f"\nTrace Statistics vs {1-alpha:.0%} Critical Values:")
    for r in range(len(trace_stat)):
        print(f"H0: rank <= {r}: Trace Stat = {trace_stat[r]:.2f}, Crit Val = {crit_vals[r]:.2f} -> {'Reject H0' if trace_stat[r] > crit_vals[r] else 'Fail to Reject H0'}")

johansen_interpretation(johansen_test)
# Based on theory for 3 variables, the maximum rank is 2.
vecm_rank = 2
print(f"\nProceeding with a cointegration rank of {vecm_rank}.")

# VECM estimation should now work without MemoryError (when we had sampled at 5mins frequency)
vecm_model = VECM(log_prices, k_ar_diff=optimal_lags-1, coint_rank=vecm_rank, deterministic='n')
vecm_results = vecm_model.fit()
print("\n--- VECM Summary ---")
print(vecm_results.summary())
print("-" * 50)


#  Post-Estimation Analysis
print("\n--- Part 3: Impulse Response and Variance Decomposition ---")
irf = var_results.irf(periods=20)
irf.plot(orth=True)
plt.suptitle('Impulse Response Functions', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

fevd = var_results.fevd(20)
print("\n--- Forecast Error Variance Decomposition ---")
print(fevd.summary())
fevd.plot()
plt.suptitle('Forecast Error Variance Decomposition', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()



## Section 3.4 : ARCH models

In [ ]:
# ==============================================================================
# Section 3.4: Volatility Modelling with GARCH and EGARCH
# ==============================================================================

warnings.filterwarnings("ignore")

# Select the asset of choice
asset_to_analyze = 'SPY5.P'
price_series = df_snp[df_snp['symbol'] == asset_to_analyze].set_index('DateTime')['price']

# Extract daily closing prices by resampling to daily frequency
daily_prices = price_series.resample('D').last().dropna()

# Compute daily percentage log returns
daily_log_returns = 100 * np.log(daily_prices / daily_prices.shift(1)).dropna()

# Demean the returns
daily_returns_demaned = daily_log_returns - daily_log_returns.mean()

print("--- Daily Data Preparation ---")
print(f"Number of daily observations: {len(daily_returns_demaned)}")
print(daily_returns_demaned.head())
print("-" * 50)


# --- Step 2: Test for ARCH Effects to Justify GARCH ---
# H0: No ARCH effects are present.
print("\n--- ARCH-LM Test on Daily Log Returns ---")
# Using statsmodels het_arch test
lm_test = het_arch(daily_returns_demaned, nlags=10)
print(f"LM Statistic: {lm_test[0]:.4f}")
print(f"p-value:      {lm_test[1]:.4f}")
if lm_test[1] < 0.05:
    print("Conclusion: Reject H0. Significant ARCH effects are present, justifying GARCH models.")
else:
    print("Conclusion: Fail to reject H0. No significant ARCH effects detected.")
print("-" * 50)


# --- Step 3: Systematically Estimate GARCH and EGARCH Models ---
# As per the assignment, we test p={1,2} and q={1,2}
p_orders = [1, 2]
q_orders = [1, 2]
garch_results = []
egarch_results = []

print("\n--- Estimating GARCH(p,q) and EGARCH(p,q) Models ---")
for p in p_orders:
    for q in q_orders:
        # GARCH model
        garch_mod = arch_model(daily_returns_demaned, vol='Garch', p=p, q=q, mean='Constant')
        res_garch = garch_mod.fit(disp='off')
        garch_results.append({'Model': f'GARCH({p},{q})', 'Log-Likelihood': res_garch.loglikelihood, 'AIC': res_garch.aic, 'BIC': res_garch.bic})
        
        # EGARCH model
        egarch_mod = arch_model(daily_returns_demaned, vol='EGarch', p=p, q=q, mean='Constant')
        res_egarch = egarch_mod.fit(disp='off')
        egarch_results.append({'Model': f'EGARCH({p},{q})', 'Log-Likelihood': res_egarch.loglikelihood, 'AIC': res_egarch.aic, 'BIC': res_egarch.bic})

# Create summary tables
garch_summary_df = pd.DataFrame(garch_results).set_index('Model')
egarch_summary_df = pd.DataFrame(egarch_results).set_index('Model')

print("\nGARCH Model Comparison:")
print(garch_summary_df)
print("\nEGARCH Model Comparison:")
print(egarch_summary_df)
print("-" * 50)


# Extract Variances and Calculate Realized Variance ---
# Find best models by BIC
best_garch_model_name = garch_summary_df['BIC'].idxmin()
best_egarch_model_name = egarch_summary_df['BIC'].idxmin()

print(f"\nBest GARCH model by BIC: {best_garch_model_name}")
print(f"Best EGARCH model by BIC: {best_egarch_model_name}")

best_garch_p, best_garch_q = eval(best_garch_model_name.replace('GARCH',''))
best_garch_fit = arch_model(daily_returns_demaned, vol='Garch', p=best_garch_p, q=best_garch_q, mean='Constant').fit(disp='off')
garch_variance = best_garch_fit.conditional_volatility**2

best_egarch_p, best_egarch_q = eval(best_egarch_model_name.replace('EGARCH',''))
best_egarch_fit = arch_model(daily_returns_demaned, vol='EGarch', p=best_egarch_p, q=best_egarch_q, mean='Constant').fit(disp='off')
egarch_variance = best_egarch_fit.conditional_volatility**2

# Calculate daily Realized Variance (RV) from intraday returns
squared_intraday_returns = clean_log_returns**2
realized_variance = squared_intraday_returns.resample('D').sum().dropna()
print("\nCalculated Realized Variance (RV) from intraday data.")
print("-" * 50)


# Plot of the Three Variance Series 
plt.figure(figsize=(15, 7))
plt.plot(garch_variance.index, garch_variance, label=f'Conditional Variance ({best_garch_model_name})')
plt.plot(egarch_variance.index, egarch_variance, label=f'Conditional Variance ({best_egarch_model_name})', linestyle='--')
plt.plot(realized_variance.index, realized_variance, label='Realized Variance (RV)', color='grey', alpha=0.8)

plt.title('Comparison of GARCH, EGARCH, and Realized Variance')
plt.ylabel('Variance')
plt.xlabel('Date')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


# --- Step 6: Feasibility of an ARMA-GARCH Model ---
# Let's demonstrate an AR(1)-GARCH(1,1) as a proof of concept.
print("\n--- Feasibility Test: Estimating an AR(1)-GARCH(1,1) Model ---")
# We use the original daily returns here, not the de-meaned ones, as the AR model will capture the mean.
try:
    arma_garch_model = arch_model(daily_log_returns, mean='AR', lags=1, vol='Garch', p=1, q=1)
    arma_garch_results = arma_garch_model.fit(disp='off')
    print("Successfully estimated an AR(1)-GARCH(1,1) model.")
    print("This demonstrates that combining a mean model (like ARMA) with a GARCH variance model is feasible.")
    print("\nAR(1)-GARCH(1,1) Summary:")
    print(arma_garch_results.summary())
except Exception as e:
    print(f"Could not estimate AR-GARCH model. Error: {e}")